# Algorithms — First Principles Reference

**AIcademy Knowledgebase | Algorithms Track**

**Sections:**
- [A-1: Big O — How to Think About Complexity](#a-1)
- [A-2: Hash Maps & Sets](#a-2)
- [A-3: Two Pointers](#a-3)
- [A-4: Sliding Window](#a-4)
- [A-5: Binary Search](#a-5)
- [A-6: BFS / DFS (Graphs & Trees)](#a-6)
- [A-7: Heap / Priority Queue](#a-7)
- [A-8: Backtracking](#a-8)
- [A-9: Dynamic Programming](#a-9)

> **How to use this notebook:** Each section gives you the mental model first, then a canonical code example, then the Big O. Cite a section by its anchor (e.g. `A-3`) when teaching or grading.


---
## A-1: Big O — How to Think About Complexity <a id='a-1'></a>

### The mental model

Big O answers one question: **as the input gets bigger, how does the work grow?** It is not "how fast does it run on my laptop." It is the shape of the growth curve as `n → ∞`.

You always state two complexities:
- **Time** — how many operations.
- **Space** — how much extra memory (not counting the input itself).

### The curves you must recognize

| Big O | Name | Example |
|---|---|---|
| O(1) | Constant | Dict/set lookup, array index |
| O(log n) | Logarithmic | Binary search |
| O(n) | Linear | Single loop over a list |
| O(n log n) | Linearithmic | Efficient sort (Timsort, Mergesort) |
| O(n²) | Quadratic | Nested loop over the same list |
| O(2ⁿ) | Exponential | Naive recursion (subsets, brute-force) |
| O(n!) | Factorial | Permutations |

### How to read your own code

```python
def has_duplicate(nums):
    for i in range(len(nums)):          # n iterations
        for j in range(i + 1, len(nums)):  # up to n iterations
            if nums[i] == nums[j]:
                return True
    return False
# Time: O(n²), Space: O(1)

def has_duplicate_fast(nums):
    seen = set()
    for n in nums:                       # n iterations
        if n in seen:                    # O(1)
            return True
        seen.add(n)                      # O(1)
    return False
# Time: O(n), Space: O(n) — trading space for time
```

### Rules of thumb

1. **Drop constants.** O(2n) → O(n).
2. **Drop non-dominant terms.** O(n² + n) → O(n²).
3. **Nested loops over the same input** → O(n²).
4. **Each step halves the search space** → O(log n).
5. **You hash/cache to skip work** → trade space for time.

> **Why this matters:** every algorithm problem in AIcademy requires you to state time and space complexity. Big O is the language of algorithm interviews.


---
## A-2: Hash Maps & Sets <a id='a-2'></a>

### The mental model

A **hash map** (Python `dict`) and **hash set** (Python `set`) give you O(1) "have I seen this?" lookups. You trade memory (you store extra stuff) for time (you skip scanning).

> The dominant move: **as you iterate once, remember what you've seen so the next item can answer questions in O(1).**

### The canonical example — Two Sum

You're given `nums = [2, 7, 11, 15]` and `target = 9`. Return the indices of the two numbers that add to the target.

**Brute force (O(n²)):**

```python
def two_sum_brute(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return (i, j)
    return None
```

**Hash map (O(n)):**

```python
def two_sum(nums, target):
    seen = {}                       # value -> index
    for i, num in enumerate(nums):
        complement = target - num   # what we NEED to have seen
        if complement in seen:
            return (seen[complement], i)
        seen[num] = i               # remember THIS number for future iterations
    return None
```

### Why subtraction, not addition?

This is the insight most beginners miss. We don't ask *"is `target + num` in the map?"* — we ask *"is `target - num` in the map?"*

We're searching for the **complement** — the missing piece that, together with the current number, sums to the target. Each iteration, the question is: *"have I already seen the number that would complete the pair?"* If yes → done. If no → add the current number so a future iteration can find it.

### Other classic hash-map moves

```python
# Frequency counting
from collections import Counter
counts = Counter("mississippi")    # {'i': 4, 's': 4, 'p': 2, 'm': 1}

# Grouping by a canonical key (e.g. anagrams)
groups = {}
for word in words:
    key = "".join(sorted(word))
    groups.setdefault(key, []).append(word)

# Set membership for O(1) "have I seen this?"
seen = set()
for x in stream:
    if x in seen:
        ...
    seen.add(x)
```

### Big O

| Operation | Time | Space |
|---|---|---|
| Lookup / insert / delete | O(1) average | — |
| Iterate | O(n) | — |
| Two Sum, group-by-key, frequency count | O(n) | O(n) |

> **When to reach for it:** "have I seen this before?", counting frequencies, grouping by a derived key, converting an O(n²) search into O(n).


---
## A-3: Two Pointers <a id='a-3'></a>

### The mental model

You walk two indices through a sequence at the same time. Two common shapes:

1. **Converging pointers** — one at the left, one at the right, moving toward each other. Usually requires the input to be sorted.
2. **Fast/slow pointers** — both start at the left; one moves faster. Used for sliding logic, cycle detection in linked lists, in-place array manipulation.

> The two-pointer trick converts many O(n²) problems into O(n) by **never re-scanning** — each pointer only ever moves forward (or only ever moves inward).

### Pair sum in a sorted array (converging)

```python
def pair_sum_sorted(nums, target):
    left, right = 0, len(nums) - 1
    while left < right:
        s = nums[left] + nums[right]
        if s == target:
            return (left, right)
        elif s < target:
            left += 1   # need a bigger sum -> move left up
        else:
            right -= 1  # need a smaller sum -> move right down
    return None
# Time: O(n), Space: O(1)
```

The reason this works: the array is sorted, so the sum at `(left, right)` is a known function of position. Increasing `left` only increases the sum; decreasing `right` only decreases it. We never need to go back.

### Remove duplicates in place (fast/slow)

```python
def remove_duplicates(nums):
    if not nums:
        return 0
    slow = 0
    for fast in range(1, len(nums)):
        if nums[fast] != nums[slow]:
            slow += 1
            nums[slow] = nums[fast]
    return slow + 1  # new length
# Time: O(n), Space: O(1)
```

`slow` marks the boundary of the de-duplicated region; `fast` scouts ahead.

### Big O

| Pattern | Time | Space |
|---|---|---|
| Converging on sorted input | O(n) | O(1) |
| Fast/slow in-place | O(n) | O(1) |

> **When to reach for it:** sorted array + pair/triplet search, palindrome checks, in-place array partitioning, or any problem where a nested loop is doing redundant work.


---
## A-4: Sliding Window <a id='a-4'></a>

### The mental model

A **sliding window** is a moving range `[left, right]` over a sequence. You expand the right edge to include new elements, and shrink the left edge when the window is "invalid." You compute a result as the window slides, without re-scanning.

Two flavors:
1. **Fixed-size window** — `right - left + 1 == k` always.
2. **Variable-size window** — `left` advances when a constraint breaks.

> Sliding window is what makes "longest/shortest substring with property X" problems O(n) instead of O(n²).

### Fixed window — max sum of k consecutive elements

```python
def max_sum_k(nums, k):
    if len(nums) < k:
        return None
    window_sum = sum(nums[:k])
    best = window_sum
    for right in range(k, len(nums)):
        window_sum += nums[right] - nums[right - k]  # add new, drop old
        best = max(best, window_sum)
    return best
# Time: O(n), Space: O(1)
```

### Variable window — longest substring without repeating characters

```python
def longest_unique_substring(s):
    seen = {}            # char -> last index
    left = 0
    best = 0
    for right, ch in enumerate(s):
        if ch in seen and seen[ch] >= left:
            left = seen[ch] + 1     # shrink window past the duplicate
        seen[ch] = right
        best = max(best, right - left + 1)
    return best
# Time: O(n), Space: O(min(n, alphabet))
```

The window stays valid (no repeats) by snapping `left` forward whenever a duplicate enters.

### Big O

| Pattern | Time | Space |
|---|---|---|
| Fixed window of size k | O(n) | O(1) |
| Variable window | O(n) | O(window content size) |

> **When to reach for it:** contiguous subarray/substring problems, "longest/shortest with property X", running averages, min/max over a moving range.


---
## A-5: Binary Search <a id='a-5'></a>

### The mental model

Binary search shrinks a search space by half on every step. Two big variants:

1. **Binary search on an index** — classic: find a value in a sorted array.
2. **Binary search on the answer** — when the question is "what's the smallest/largest X that satisfies a condition?" and the condition is monotonic, search the answer space, not the input.

> The hard part is not the loop — it's getting `left`, `right`, and the boundary update right. Pick a template and stick with it.

### Classic binary search

```python
def binary_search(nums, target):
    left, right = 0, len(nums) - 1   # inclusive range
    while left <= right:
        mid = (left + right) // 2
        if nums[mid] == target:
            return mid
        elif nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1
# Time: O(log n), Space: O(1)
```

### Binary search on the answer — Koko eating bananas

You have piles of bananas and `h` hours. Each hour you eat up to `k` from one pile. Find the **minimum** `k` that lets you finish in time.

```python
import math

def min_eating_speed(piles, h):
    def can_finish(k):
        return sum(math.ceil(p / k) for p in piles) <= h

    left, right = 1, max(piles)
    while left < right:
        mid = (left + right) // 2
        if can_finish(mid):
            right = mid          # try smaller k
        else:
            left = mid + 1       # need bigger k
    return left
# Time: O(n log max(piles)), Space: O(1)
```

We're not searching the array — we're searching the *answer space* `[1, max(piles)]` for the smallest `k` where `can_finish(k)` flips from False to True.

### Big O

| Pattern | Time | Space |
|---|---|---|
| On a sorted array | O(log n) | O(1) |
| On the answer (predicate has cost C) | O(C · log(answer range)) | O(1) |

> **When to reach for it:** sorted input, "find min/max X such that...", or any monotonic predicate over a range of integers.


---
## A-6: BFS / DFS (Graphs & Trees) <a id='a-6'></a>

### The mental model

You're exploring connected nodes. Two strategies:

- **BFS (breadth-first)** — explore in layers, nearest neighbors first. Uses a **queue**. Finds the **shortest path in an unweighted graph**.
- **DFS (depth-first)** — go as deep as possible before backtracking. Uses a **stack** (or recursion). Natural for **path enumeration, cycle detection, topological sort**.

> The bug everyone hits: **forgetting to mark nodes visited.** You'll revisit forever.

### BFS — number of islands

```python
from collections import deque

def num_islands(grid):
    if not grid:
        return 0
    rows, cols = len(grid), len(grid[0])
    visited = set()
    count = 0

    def bfs(r, c):
        q = deque([(r, c)])
        visited.add((r, c))
        while q:
            x, y = q.popleft()
            for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                nx, ny = x + dx, y + dy
                if (0 <= nx < rows and 0 <= ny < cols
                    and grid[nx][ny] == "1" and (nx, ny) not in visited):
                    visited.add((nx, ny))
                    q.append((nx, ny))

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1" and (r, c) not in visited:
                bfs(r, c)
                count += 1
    return count
# Time: O(rows * cols), Space: O(rows * cols) for visited
```

### DFS — recursive tree traversal

```python
def inorder(root):
    if not root:
        return []
    return inorder(root.left) + [root.val] + inorder(root.right)
# Time: O(n), Space: O(h) recursion stack where h = tree height
```

### When BFS vs DFS

| Use BFS when... | Use DFS when... |
|---|---|
| You want the shortest path | You want all paths / any path |
| You want layer-by-layer order | You're checking for cycles |
| Unweighted graph | Tree traversal (pre/in/post-order) |
| | Topological sort |

### Big O

| Operation | Time | Space |
|---|---|---|
| BFS / DFS on graph with V nodes, E edges | O(V + E) | O(V) |

> **When to reach for it:** grid problems, connected components, shortest path in unweighted graphs, tree traversal, cycle detection, topological order.


---
## A-7: Heap / Priority Queue <a id='a-7'></a>

### The mental model

A **heap** is a tree structure where the smallest (min-heap) or largest (max-heap) element is always on top. In Python, `heapq` is a **min-heap** on a list.

> A heap gives you O(log n) push and O(log n) pop of the extreme element. Use it when you need to repeatedly answer "what's the smallest/largest so far?"

### Top-K largest elements

```python
import heapq

def top_k_largest(nums, k):
    # Keep a min-heap of size k. The smallest in the heap is the kth largest overall.
    heap = []
    for n in nums:
        heapq.heappush(heap, n)
        if len(heap) > k:
            heapq.heappop(heap)
    return sorted(heap, reverse=True)
# Time: O(n log k), Space: O(k)
```

Compared to sorting the whole list (O(n log n)), this is faster when k << n.

### Max-heap trick

Python's `heapq` is min-only. To get a max-heap, push the **negated** value:

```python
heapq.heappush(heap, -n)
largest = -heapq.heappop(heap)
```

### Merge K sorted lists

```python
import heapq

def merge_k_sorted(lists):
    heap = []
    for i, lst in enumerate(lists):
        if lst:
            heapq.heappush(heap, (lst[0], i, 0))   # (value, list_idx, elem_idx)
    result = []
    while heap:
        val, i, j = heapq.heappop(heap)
        result.append(val)
        if j + 1 < len(lists[i]):
            heapq.heappush(heap, (lists[i][j+1], i, j+1))
    return result
# Time: O(N log k) where N = total elements, k = number of lists
```

### Big O

| Operation | Time |
|---|---|
| Push | O(log n) |
| Pop min/max | O(log n) |
| Peek min/max | O(1) |
| Heapify a list of n | O(n) |

> **When to reach for it:** top-k, k-way merge, scheduling/priority problems, running median (two heaps), Dijkstra's shortest path.


---
## A-8: Backtracking <a id='a-8'></a>

### The mental model

Backtracking is **DFS through a tree of choices**. You pick an option, recurse, and if that path doesn't work (or after you've fully explored it), you **undo the choice** and try another.

Every backtracking solution has the same shape:

```python
def backtrack(state, choices):
    if is_complete(state):
        record(state)
        return
    for choice in choices:
        if is_valid(choice, state):
            apply(choice, state)
            backtrack(state, remaining_choices)
            undo(choice, state)        # the "backtrack" step
```

> The "undo" step is what makes it backtracking. Without it, you're just doing recursion with bugs.

### Generate all permutations

```python
def permutations(nums):
    result = []

    def backtrack(current, remaining):
        if not remaining:
            result.append(current[:])
            return
        for i in range(len(remaining)):
            current.append(remaining[i])
            backtrack(current, remaining[:i] + remaining[i+1:])
            current.pop()              # undo

    backtrack([], nums)
    return result
# Time: O(n · n!), Space: O(n) recursion depth
```

### Subsets

```python
def subsets(nums):
    result = []

    def backtrack(start, current):
        result.append(current[:])
        for i in range(start, len(nums)):
            current.append(nums[i])
            backtrack(i + 1, current)
            current.pop()              # undo

    backtrack(0, [])
    return result
# Time: O(n · 2ⁿ), Space: O(n)
```

### N-Queens (constraint-based)

```python
def solve_n_queens(n):
    cols, diag1, diag2 = set(), set(), set()
    board = [["."] * n for _ in range(n)]
    results = []

    def backtrack(row):
        if row == n:
            results.append(["".join(r) for r in board])
            return
        for col in range(n):
            if col in cols or (row - col) in diag1 or (row + col) in diag2:
                continue
            cols.add(col); diag1.add(row - col); diag2.add(row + col)
            board[row][col] = "Q"
            backtrack(row + 1)
            cols.remove(col); diag1.remove(row - col); diag2.remove(row + col)
            board[row][col] = "."

    backtrack(0)
    return results
```

### Big O

Backtracking is usually exponential in the worst case (you're exploring a decision tree). Pruning (`is_valid` checks) is what makes it tractable.

| Problem | Time |
|---|---|
| Permutations of n | O(n · n!) |
| Subsets of n | O(n · 2ⁿ) |
| N-Queens | O(n!) worst case, much less with pruning |

> **When to reach for it:** generate all X (permutations, combinations, subsets), constraint satisfaction (Sudoku, N-Queens), word search on a grid, parenthesis generation.


---
## A-9: Dynamic Programming <a id='a-9'></a>

### The mental model

DP works when a problem has two properties:

1. **Optimal substructure** — the answer to the big problem can be built from answers to smaller subproblems.
2. **Overlapping subproblems** — the same subproblem appears many times in the naive recursion.

You either **memoize** (top-down: recursion + cache) or **tabulate** (bottom-up: fill an array).

> The hardest part is **defining the state.** What does `dp[i]` mean? Once that's clear, the transition usually writes itself.

### Climbing stairs (1D DP)

You can take 1 or 2 steps at a time. How many ways to climb `n` stairs?

```python
def climb_stairs(n):
    if n <= 2:
        return n
    dp = [0] * (n + 1)
    dp[1], dp[2] = 1, 2
    for i in range(3, n + 1):
        dp[i] = dp[i-1] + dp[i-2]   # came from one step below OR two steps below
    return dp[n]
# Time: O(n), Space: O(n) — easily reducible to O(1)
```

State: `dp[i]` = number of ways to reach step `i`. Transition: arrived from step `i-1` (1 step) or step `i-2` (2 steps).

### Coin change (minimization)

Fewest coins to make amount `n` from denominations `coins`.

```python
def coin_change(coins, n):
    INF = float("inf")
    dp = [INF] * (n + 1)
    dp[0] = 0
    for amount in range(1, n + 1):
        for c in coins:
            if c <= amount:
                dp[amount] = min(dp[amount], dp[amount - c] + 1)
    return dp[n] if dp[n] != INF else -1
# Time: O(n · len(coins)), Space: O(n)
```

State: `dp[amount]` = fewest coins to make `amount`. Transition: try each coin and take the best.

### Longest common subsequence (2D DP)

```python
def lcs(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]
# Time: O(m·n), Space: O(m·n)
```

### How to find the recurrence

1. **Define state.** "Let `dp[i]` mean..."
2. **Define the answer.** Usually `dp[n]` or `max(dp)` or `dp[m][n]`.
3. **Find the transition.** "To compute `dp[i]`, I look at smaller states `dp[i-1]`, `dp[i-k]`, etc."
4. **Define base cases.** What is `dp[0]` (or `dp[0][j]`)?
5. **Decide direction.** Top-down (memoize) or bottom-up (tabulate).

### Big O

DP transforms exponential brute force (e.g. O(2ⁿ) recursion) into polynomial — usually **O(states · work per state)**.

| Problem | States | Work | Total |
|---|---|---|---|
| Climbing stairs | n | O(1) | O(n) |
| Coin change | n | O(coins) | O(n · coins) |
| Longest common subsequence | m·n | O(1) | O(m·n) |
| 0/1 knapsack | n·W | O(1) | O(n·W) |

> **When to reach for it:** "count the ways," "min/max cost," "longest/shortest," or any problem where naive recursion repeats the same subproblems. Look for a *small set of changing parameters* that defines the state.
